# Full-Stack Integration: Redis + OpenAI Agents SDK

This notebook demonstrates all Redis components working together to build a production-ready AI agent.

**Components:**
- `AgentSession` - Persistent conversation memory
- `SemanticCache` - Reduce LLM calls with intelligent caching
- `RedisVectorStore` - RAG document search
- `RedisFullTextSearch` - BM25 keyword search
- `RedisStreamTransport` - Reliable token streaming
- `AgentMetrics` - Time-series observability

## Setup

In [1]:
import os
import time
from dotenv import load_dotenv

load_dotenv()

REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://redis:6379


In [2]:
# Import all Redis components
from redis_openai_agents import (
    AgentSession,
    SemanticCache,
    RedisVectorStore,
    RedisFullTextSearch,
    RedisStreamTransport,
    AgentMetrics,
)

print("All components imported successfully!")

All components imported successfully!


## 1. Initialize All Components

In [3]:
# Session for conversation memory
session = AgentSession(
    user_id="demo_user",
    conversation_id="demo_conversation",
    redis_url=REDIS_URL,
)
print(f"Session: {session.conversation_id}")

# Cache for LLM response caching
cache = SemanticCache(
    redis_url=REDIS_URL,
    similarity_threshold=0.95,
    ttl=3600,
    name="demo_cache",
)
print("Cache: initialized")

# Vector store for RAG
vector_store = RedisVectorStore(
    name="demo_knowledge",
    redis_url=REDIS_URL,
)
print("Vector Store: initialized")

# Full-text search for keyword matching
fts = RedisFullTextSearch(
    name="demo_articles",
    redis_url=REDIS_URL,
)
print("Full-Text Search: initialized")

# Stream for token delivery
stream = RedisStreamTransport(
    stream_name="demo_stream",
    redis_url=REDIS_URL,
)
print(f"Stream: {stream.stream_name}")

# Metrics for observability
metrics = AgentMetrics(
    name="demo_agent",
    redis_url=REDIS_URL,
)
print(f"Metrics: {metrics.name}")

10:52:35 redisvl.index.index INFO   Index already exists, not overwriting.


Session: demo_conversation


10:52:40 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:52:40 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


Cache: initialized
10:52:43 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:52:43 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


Vector Store: initialized
Full-Text Search: initialized
Stream: demo_stream
Metrics: demo_agent


## 2. Populate Knowledge Base

In [4]:
# Add documents to vector store for RAG
knowledge_docs = [
    {
        "content": "Redis is an open-source, in-memory data structure store used as a database, cache, and message broker.",
        "metadata": {"topic": "redis", "type": "overview"}
    },
    {
        "content": "Redis supports data structures such as strings, hashes, lists, sets, sorted sets, bitmaps, and streams.",
        "metadata": {"topic": "redis", "type": "features"}
    },
    {
        "content": "Redis 8 ships with JSON, search, time series, and probabilistic data structures built in.",
        "metadata": {"topic": "redis", "type": "overview"}
    },
    {
        "content": "The OpenAI Agents SDK provides a framework for building AI agents with tools, handoffs, and guardrails.",
        "metadata": {"topic": "agents", "type": "overview"}
    },
]

vector_ids = vector_store.add_documents(knowledge_docs)
print(f"Added {len(vector_ids)} documents to vector store")

# Add articles to full-text search
articles = [
    {
        "title": "Getting Started with Redis",
        "content": "Redis is a powerful in-memory data store. Learn how to install and configure Redis for your applications.",
        "category": "tutorial",
        "tags": ["redis", "beginner"]
    },
    {
        "title": "Building AI Agents with OpenAI SDK",
        "content": "The OpenAI Agents SDK makes it easy to build intelligent agents. This guide covers tools, handoffs, and memory.",
        "category": "tutorial",
        "tags": ["ai", "agents", "openai"]
    },
    {
        "title": "Redis Vector Search Deep Dive",
        "content": "Explore Redis vector search capabilities for building RAG applications and semantic search systems.",
        "category": "advanced",
        "tags": ["redis", "vector", "rag"]
    },
]

fts_ids = fts.add_documents(articles)
print(f"Added {len(fts_ids)} articles to full-text search")

Added 4 documents to vector store
Added 3 articles to full-text search


## 3. Simulate Agent Request with All Components

In [5]:
def process_query(query: str) -> str:
    """Process a user query using all components."""
    start_time = time.time()
    cache_hit = False
    
    # Step 1: Check semantic cache
    cached = cache.get(query=query)
    if cached:
        cache_hit = True
        response = cached.response
        print(f"  Cache HIT (similarity: {cached.similarity:.2f})")
    else:
        print("  Cache MISS - gathering context...")
        
        # Step 2: Search knowledge base (vector search)
        vector_results = vector_store.search(query=query, k=2)
        print(f"  Vector search: {len(vector_results)} relevant docs")
        
        # Step 3: Search articles (full-text search)
        fts_results = fts.search(query=query, k=2)
        print(f"  Full-text search: {len(fts_results)} matching articles")
        
        # Step 4: Build context for LLM (simulated)
        context_parts = []
        for doc in vector_results:
            context_parts.append(doc.get("content", ""))
        for article in fts_results:
            context_parts.append(f"{article['title']}: {article['content']}")
        
        # Simulated LLM response (in production, would call actual LLM)
        response = f"Based on my knowledge: {context_parts[0] if context_parts else 'No context found'}"
        
        # Step 5: Cache the response
        cache.set(query=query, response=response)
        print("  Response cached")
    
    # Step 6: Store in session history
    session.add_message(role="user", content=query)
    session.add_message(role="assistant", content=response)
    
    # Step 7: Publish to stream
    stream.publish(
        event_type="response",
        data={"query": query, "response": response[:100]}
    )
    
    # Step 8: Record metrics
    latency = (time.time() - start_time) * 1000
    metrics.record(
        latency_ms=latency,
        input_tokens=len(query.split()),
        output_tokens=len(response.split()),
        cache_hit=cache_hit,
    )
    print(f"  Latency: {latency:.1f}ms")
    
    return response

In [6]:
# Process some queries
queries = [
    "What is Redis?",
    "Tell me about Redis",  # Should hit cache (semantic match)
    "How do I build AI agents?",
    "What data structures does Redis support?",
]

print("Processing queries:\n")
for query in queries:
    print(f"Query: {query}")
    response = process_query(query)
    print(f"Response: {response[:80]}...\n")

Processing queries:

Query: What is Redis?
  Cache MISS - gathering context...
  Vector search: 2 relevant docs
  Full-text search: 0 matching articles


  Response cached
  Latency: 209.3ms
Response: Based on my knowledge: Redis is an open-source, in-memory data structure store u...

Query: Tell me about Redis
  Cache MISS - gathering context...
  Vector search: 2 relevant docs
  Full-text search: 0 matching articles


  Response cached
  Latency: 208.1ms
Response: Based on my knowledge: Redis is an open-source, in-memory data structure store u...

Query: How do I build AI agents?
  Cache MISS - gathering context...
  Vector search: 2 relevant docs
  Full-text search: 0 matching articles


  Response cached
  Latency: 242.1ms
Response: Based on my knowledge: The OpenAI Agents SDK provides a framework for building A...

Query: What data structures does Redis support?
  Cache MISS - gathering context...
  Vector search: 2 relevant docs
  Full-text search: 0 matching articles
  Response cached
  Latency: 177.9ms
Response: Based on my knowledge: Redis supports data structures such as strings, hashes, l...



## 4. View Component Statistics

In [7]:
# Session stats
print("Session Statistics")
print("=" * 40)
print(f"Messages: {session.message_count()}")
print()

# Cache stats (with L1/L2 breakdown)
cache_stats = cache.get_stats()
print("Cache Statistics")
print("=" * 40)
print(f"Total Hits: {cache_stats['hits']}")
print(f"  - L1 (exact match): {cache_stats['l1_hits']}")
print(f"  - L2 (semantic): {cache_stats['l2_hits']}")
print(f"Misses: {cache_stats['misses']}")
total = cache_stats['hits'] + cache_stats['misses']
if total > 0:
    print(f"Hit Rate: {cache_stats['hits'] / total * 100:.1f}%")
print()

# Vector store stats
print("Vector Store Statistics")
print("=" * 40)
print(f"Documents: {vector_store.count()}")
print()

# Full-text search stats
print("Full-Text Search Statistics")
print("=" * 40)
print(f"Documents: {fts.count()}")
print()

# Stream stats
stream_info = stream.info()
print("Stream Statistics")
print("=" * 40)
print(f"Length: {stream_info['length']}")
print(f"Consumer Groups: {stream_info['groups']}")
print()

# Metrics stats
metrics_stats = metrics.get_stats()
print("Agent Metrics")
print("=" * 40)
print(f"Total Requests: {metrics_stats['count']}")
print(f"Avg Latency: {metrics_stats['latency_avg']:.2f}ms")
print(f"Cache Hit Rate: {metrics_stats['cache_hit_rate']:.1%}")

Session Statistics
Messages: 8

Cache Statistics
Total Hits: 0
  - L1 (exact match): 0
  - L2 (semantic): 0
Misses: 4
Hit Rate: 0.0%

Vector Store Statistics
Documents: 4

Full-Text Search Statistics
Documents: 3

Stream Statistics
Length: 4
Consumer Groups: 1

Agent Metrics
Total Requests: 4
Avg Latency: 209.35ms
Cache Hit Rate: 0.0%


## 5. Cleanup

In [8]:
# Clean up all demo data
session.delete()
cache.clear()
vector_store.delete_all()
fts.delete_all()
stream.delete()
metrics.delete()

print("All demo data cleaned up!")

10:52:46 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:52:46 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


All demo data cleaned up!


## Summary

This notebook demonstrated how all Redis components work together:

| Component | Redis Feature | Purpose |
|-----------|---------------|----------|
| `AgentSession` | Hash + JSON | Conversation memory |
| `SemanticCache` | Vector Search | LLM response caching |
| `RedisVectorStore` | Vector Search (HNSW) | RAG document search |
| `RedisFullTextSearch` | FT.SEARCH (BM25) | Keyword search |
| `RedisStreamTransport` | Streams | Token streaming |
| `AgentMetrics` | TimeSeries | Observability |

**Benefits:**
- Unified data platform (no separate databases)
- Low latency (all in-memory)
- Reduced LLM costs (semantic caching)
- Production-ready observability